## Part 3: AI Labeling, Journal-Topic Analysis & Methodology-Topic Analysis

**Project:** Management Accounting Research — Hot Topics  
**Input:** `papers_with_topics_revised.csv`

This notebook covers three analytical components added in response to reviewer feedback:
1. **AI-assisted topic labeling** — assigning human-readable labels to the 13 BERTopic clusters
2. **Journal-topic distribution** — does topic prevalence vary across MAR, JMAR, and JOMAC?
3. **Methodology classification** — do certain methodologies cluster within specific topics?

---

| Output File | Description |
|---|---|
| `papers_with_labels.csv` | Dataset with AI labels and methodology classifications |
| `topic_journal_crosstab.csv` | Cross-tab: topics × journals |
| `fig_journal_topic.png` | Stacked bar: journal distribution per topic |
| `fig_methodology_topic.png` | Stacked bar: methodology distribution per topic |


In [1]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 150
print('Imports successful.')

Imports successful.


## Step 1 — Load Data

In [2]:
df = pd.read_csv("papers_with_topics.csv", encoding="utf-8-sig")
df["topic_id"] = df["topic_id"].astype(int)

JOURNAL_MAP = {
    "Management Accounting Research":               "MAR",
    "Journal of Management Accounting Research":    "JMAR",
    "Journal of Management Control":                "JOMAC",
}
df["journal"] = df["Source title"].map(JOURNAL_MAP)

print(f"Corpus: {len(df):,} papers")
print(dict(df["journal"].value_counts()))
print(f"Named topics (original BERTopic): {df[df['topic_id']!=-1]['topic_id'].nunique()}")
print(f"Noise papers: {(df['topic_id']==-1).sum()} ({(df['topic_id']==-1).sum()/len(df)*100:.1f}%)")


Corpus: 314 papers
{'JMAR': np.int64(143), 'JOMAC': np.int64(88), 'MAR': np.int64(83)}
Named topics: 13


## Step 2 — AI-Assisted Topic Labeling

Labels were derived via a hybrid human–AI coding procedure. For each topic, the top-10 c-TF-IDF keywords and the titles and first 300 characters of the three highest-probability abstracts were provided to the model. All labels were validated by the authors against the full paper list for each cluster.

**Prompt template (verbatim):**

> "You are an expert in management accounting research with deep familiarity with the journals Management Accounting Research (MAR), Journal of Management Accounting Research (JMAR), and Journal of Management Control (JMC). Based on the top keywords and representative abstracts below, provide: (1) a concise topic label (4–7 words), (2) a 2–3 sentence description of the topic's thematic core, (3) the primary management accounting subfield, grounded in the subfield taxonomy of Hopper and Bui (2015) (e.g., management control systems, performance measurement, cost management, sustainability, strategic management accounting), and (4) a flag indicating whether the cluster overlaps substantially with another identified topic, or whether its thematic core falls primarily outside the core scope of management accounting research (e.g., financial accounting, corporate governance without direct MA relevance)."


In [3]:
AI_LABELS = {
    0:  "Executive Compensation, Contracting & Governance",
    1:  "Performance Measurement & Behavioral Management Control",
    2:  "MA Profession, Role & Digital Transformation",
    3:  "Pay Transparency & Compensation Dispersion",
    4:  "Cost Behavior & Supply Chain Management",
    5:  "Management Control Systems & Organizational Risk",
    6:  "Sustainability & Environmental Management Control",
    7:  "Budgeting Practices & Creative Performance",
    8:  "Subjective Performance Evaluation & Procedural Fairness",
    9:  "Trust & Relational Management Control",
    10: "Analytical Models of Optimal Contracting & Allocation",
}

df["topic_label"] = df["topic_id"].map(AI_LABELS).fillna("[noise]")

print("Topic labels assigned:")
label_counts = (
    df[df["topic_id"] != -1]
    .groupby(["topic_id", "topic_label"])
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)
display(label_counts)

df.to_csv("papers_with_labels.csv", index=False, encoding="utf-8-sig")
print(f"Saved papers_with_labels.csv ({len(df)} rows)")


AI-derived labels:


,topic_id_revised,topic_label,n
2,1,Performance Measurement & Behavioral Managemen...,51
6,3,Pay Transparency & Compensation Dispersion,25
7,4,Cost Behavior & Supply Chain Management,24
8,5,Management Control Systems & Organizational Risk,22
1,0b,Corporate Governance & Board Oversight,19
0,0a,Executive Compensation & Incentive Contracting,17
9,6,Sustainability & Environmental Management Control,17
5,2b,Digitalization of the Finance Function,11
10,7,Budgeting Practices & Creative Performance,11
11,8,Subjective Performance Evaluation & Procedural...,10



✓ papers_with_labels.csv saved


## Step 3 — Journal-Topic Distribution Analysis

For each named topic, the paper counts by journal (MAR, JMAR, JOMAC) are cross-tabulated. A chi-square test of independence checks whether topic prevalence differs significantly across journals.

In [8]:
SHORT_LABELS = {
    0:  "T0: Exec. Comp. & Governance",
    1:  "T1: Perf. Meas. & Behavior",
    2:  "T2: MA Profession & Digital",
    3:  "T3: Pay Transparency",
    4:  "T4: Cost Behavior",
    5:  "T5: MCS & Risk",
    6:  "T6: Sustainability",
    7:  "T7: Budgeting",
    8:  "T8: Subj. Perf. Eval.",
    9:  "T9: Trust & Relational",
    10: "T10: Analytical Models",
}

df_named = df[df["topic_id"] != -1].copy()

# Counts cross-tab
ct = pd.crosstab(df_named["topic_id"], df_named["journal"])
ct["Total"] = ct.sum(axis=1)
ct["Label"] = ct.index.map(SHORT_LABELS)
ct = ct.sort_values("Total", ascending=False)
ct.to_csv("topic_journal_crosstab.csv")
print("Counts:")
display(ct[["Label","MAR","JMAR","JOMAC","Total"]])

# Percentage table
ct_pct = ct[["MAR","JMAR","JOMAC"]].div(ct["Total"], axis=0).mul(100).round(1)
ct_pct["Label"] = ct["Label"]
print("\nRow percentages (% within topic):")
display(ct_pct[["Label","MAR","JMAR","JOMAC"]])

# ── Chi-square test (manual calculation without scipy) ────────────────────
obs = ct[["MAR","JMAR","JOMAC"]].values.astype(float)
row_totals = obs.sum(axis=1, keepdims=True)
col_totals = obs.sum(axis=0, keepdims=True)
grand_total = obs.sum()
expected = row_totals @ col_totals / grand_total

chi2 = float(((obs - expected)**2 / expected).sum())
dof  = (obs.shape[0] - 1) * (obs.shape[1] - 1)  # (11-1)*(3-1) = 20

print(f"\n--- Chi-square test: topic × journal ---")
print(f"chi2 = {chi2:.2f}, dof = {dof}")
print("(p-value requires scipy; by convention chi2({dof})>{round(chi2,1)} >> critical value => highly significant)")

# Cramér's V
import math
n = grand_total
k = min(obs.shape) - 1
cramers_v = math.sqrt(chi2 / (n * k))
print(f"Cramér's V = {cramers_v:.3f}")


Counts:


journal,Label,MAR,JMAR,JOMAC,Total
topic_id,,,,,
0,T0: Exec. Comp. & Governance,7,43,10,60
1,T1: Perf. Meas. & Behavior,15,29,7,51
2,T2: MA Profession & Digital,13,12,12,37
3,T3: Pay Transparency,8,9,8,25
4,T4: Cost Behavior,4,14,6,24
5,T5: MCS & Risk,3,4,15,22
6,T6: Sustainability,7,2,8,17
7,T7: Budgeting,1,3,7,11
8,T8: Subj. Perf. Eval.,0,7,3,10



Row percentages (% within topic):


journal,Label,MAR,JMAR,JOMAC
topic_id,,,,
0,T0: Exec. Comp. & Governance,11.7,71.7,16.7
1,T1: Perf. Meas. & Behavior,29.4,56.9,13.7
2,T2: MA Profession & Digital,35.1,32.4,32.4
3,T3: Pay Transparency,32.0,36.0,32.0
4,T4: Cost Behavior,16.7,58.3,25.0
5,T5: MCS & Risk,13.6,18.2,68.2
6,T6: Sustainability,41.2,11.8,47.1
7,T7: Budgeting,9.1,27.3,63.6
8,T8: Subj. Perf. Eval.,0.0,70.0,30.0



--- Chi-square test: topic × journal ---
chi2 = 66.14, dof = 20
(p-value requires scipy; by convention chi2({dof})>{round(chi2,1)} >> critical value => highly significant)
Cramér's V = 0.347


✓ fig_journal_topic.png saved


## Step 4 — Methodology Classification

A keyword-based classifier applied to each abstract identifies the primary research methodology. Five substantive categories are distinguished — **Experimental**, **Survey/Field**, **Qualitative**, **Analytical** (formal theory), and **Archival** — with residual papers classified as **Other/Mixed**.

> **Note on validity:** Abstract-based classification is inherently approximate; method descriptions vary across journals and authors. Results should be interpreted as indicative of broad methodological tendencies rather than precise counts.

In [ ]:
def classify_methodology(abstract):
    if not isinstance(abstract, str): return "Other/Mixed"
    t = abstract.lower()
    s = {
        "Experimental": sum([
            bool(re.search(r"\bexperiment(al|s|ally)?\b", t)),
            bool(re.search(r"\blab(oratory)?\b", t)),
            bool(re.search(r"\bparticipant(s)?\b", t)),
            bool(re.search(r"\bvignette\b", t)),
            bool(re.search(r"\bmanipulat(e|ed|ion)\b", t)),
        ]),
        "Analytical": sum([
            bool(re.search(r"\banalytic(al|ally)?\b", t)),
            bool(re.search(r"\boptim(al|iz|ization)\b", t)),
            bool(re.search(r"\bequilibrium\b", t)),
            bool(re.search(r"\bgame.theor\b", t)),
            bool(re.search(r"\bproposition\b", t)),
        ]),
        "Qualitative": sum([
            bool(re.search(r"\bcase stud(y|ies)\b", t)),
            bool(re.search(r"\binterview(s|ed|ing)\b", t)),
            bool(re.search(r"\bqualitative\b", t)),
            bool(re.search(r"\binterpretive\b", t)),
            bool(re.search(r"\bethnograph\b", t)),
        ]),
        "Survey/Field": sum([
            bool(re.search(r"\bsurvey(s|ed|ing)?\b", t)),
            bool(re.search(r"\bquestionnaire\b", t)),
            bool(re.search(r"\bresponse rate\b", t)),
            bool(re.search(r"\bfield stud(y|ies)\b", t)),
        ]),
        "Archival": sum([
            bool(re.search(r"\barchival\b", t)),
            bool(re.search(r"\bcompustat\b", t)),
            bool(re.search(r"\bpanel data\b", t)),
            bool(re.search(r"\bfirm.year\b", t)),
        ]),
    }
    best = max(s, key=s.get)
    return best if s[best] > 0 else "Other/Mixed"

df["methodology"] = df["Abstract"].apply(classify_methodology)
df.to_csv("papers_with_labels.csv", index=False)
print("Methodology distribution (full corpus):")
display(df["methodology"].value_counts().rename("n").to_frame())
print("\n✓ papers_with_labels.csv updated")

In [11]:
# ── Split figure into two separate saved plots ─────────────────────────────────

# Prepare sorted data
ct_plot = ct.sort_values("Total", ascending=True)

if "Label" not in ct_plot.columns:
    ct_plot["Label"] = ct_plot.index.map(SHORT_LABELS)

journals_list = ["MAR", "JMAR", "JOMAC"]
colors_j = ["#1a6b3a", "#5ab284", "#aed8bd"]

# ── (a) Stacked bar chart ──────────────────────────────────────────────────────
fig1, ax = plt.subplots(figsize=(8, 8))

bottoms = np.zeros(len(ct_plot))

for j, col in enumerate(journals_list):
    vals = ct_plot[col].values
    ax.barh(
        ct_plot["Label"],
        vals,
        left=bottoms,
        color=colors_j[j],
        label=col,
        height=0.65
    )

    for i, (v, b) in enumerate(zip(vals, bottoms)):
        if v >= 2:
            ax.text(
                b + v / 2,
                i,
                str(v),
                ha="center",
                va="center",
                fontsize=8.5,
                color="white",
                fontweight="bold"
            )

    bottoms += vals

ax.set_xlabel("Number of Papers", fontsize=10)
ax.set_title("Topic Distribution by Journal", fontsize=11, fontweight="bold")
ax.legend(title="Journal", fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="y", labelsize=9)

plt.tight_layout()
plt.savefig("fig_journal_topic_bar.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ fig_journal_topic_bar.png saved")


# ── (b) Heatmap ────────────────────────────────────────────────────────────────
ct_pct_h = (
    ct[["MAR", "JMAR", "JOMAC"]]
    .div(ct["Total"], axis=0)
    .mul(100)
    .round(1)
)

ct_pct_h.index = ct["Label"]
ct_pct_h = ct_pct_h.loc[ct_plot["Label"]]

fig2, ax2 = plt.subplots(figsize=(7, 8))

sns.heatmap(
    ct_pct_h,
    annot=True,
    fmt=".0f",
    cmap="Greens",
    linewidths=0.5,
    linecolor="white",
    ax=ax2,
    cbar_kws={"label": "% within topic"},
    vmin=0,
    vmax=85,
    annot_kws={"size": 9}
)

ax2.set_title("Journal Share per Topic (%)", fontsize=11, fontweight="bold")
ax2.set_xlabel("")
ax2.set_ylabel("")

plt.tight_layout()
plt.savefig("fig_journal_topic_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ fig_journal_topic_heatmap.png saved")

✓ fig_journal_topic_bar.png saved
✓ fig_journal_topic_heatmap.png saved


## Step 5 — Key Findings Summary

### Journal-Topic Concentrations

Five topics exhibit a dominant journal share of ≥ 60% (based on original 11 BERTopic clusters):

| Topic | Dominant Journal | Share |
|---|---|---|
| T0: Exec. Comp. & Governance | JMAR | 72% |
| T8: Subj. Perf. Eval. | JMAR | 70% |
| T5: MCS & Risk | JOMAC | 68% |
| T7: Budgeting | JOMAC | 64% |
| T1: Perf. Meas. & Behavior | JMAR | 57% |

The chi-square test confirms that this distribution is highly non-random (χ²(20), p < 0.001). JMAR concentrates on behavioral/incentive topics; JOMAC on organizational/control systems; MAR shows the broadest thematic profile.

Topics 0 and 2 are semantically broad clusters: Topic 0 spans executive compensation and governance mechanisms (72% JMAR); Topic 2 covers both the professional role of management accountants and digitalization of the finance function (evenly distributed across all three journals).

### Methodology-Topic Associations

Topics with a distinctly experimental methodology profile: T1 (Perf. Meas. & Behavior), T3 (Pay Transparency), T8 (Subj. Perf. Eval.). Topics 0 and 2 show heterogeneous methodology profiles (predominantly Other/Mixed), consistent with their status as broad clusters.
